In [1]:
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

# Define the path to the main directory containing subdirectories for each class
data_directory = 'data/'

# Specify the image dimensions and batch size
img_width, img_height = 128, 128
batch_size = 32

In [2]:
# Use ImageDataGenerator for data augmentation and normalization
datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # Adjust the validation split ratio as needed
)

# Create separate generators for training and validation sets
train_generator = datagen.flow_from_directory(
    data_directory,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical',
    classes=['unidentified', 'Hyalomma', 'Rhipicephalus'],
    subset='training'  # Specify 'training' for the training set
)

validation_generator = datagen.flow_from_directory(
    data_directory,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical',
    classes=['unidentified', 'Hyalomma', 'Rhipicephalus'],
    subset='validation'  # Specify 'validation' for the validation set
)

Found 13108 images belonging to 3 classes.
Found 3275 images belonging to 3 classes.


In [3]:
# Build a simple CNN model
model = Sequential()
model.add(Conv2D(32, (3, 3), input_shape=(img_width, img_height, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(3, activation='softmax'))  # Assuming you have 3 classes

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [5]:
# Train the model using the generators
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    epochs=20,  # Adjust the number of epochs as needed
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // batch_size
)

Epoch 1/20
409/409 [==============================] - 254s 622ms/step - loss: 0.1284 - accuracy: 0.9563 - val_loss: 1.4395 - val_accuracy: 0.5236
Epoch 2/20
409/409 [==============================] - 260s 635ms/step - loss: 0.0530 - accuracy: 0.9815 - val_loss: 2.0000 - val_accuracy: 0.4807
Epoch 3/20
409/409 [==============================] - 262s 640ms/step - loss: 0.0292 - accuracy: 0.9907 - val_loss: 1.3972 - val_accuracy: 0.6268
Epoch 4/20
409/409 [==============================] - 254s 620ms/step - loss: 0.0265 - accuracy: 0.9915 - val_loss: 0.6135 - val_accuracy: 0.8100
Epoch 5/20
409/409 [==============================] - 263s 642ms/step - loss: 0.0178 - accuracy: 0.9940 - val_loss: 0.8444 - val_accuracy: 0.7258
Epoch 6/20
409/409 [==============================] - 256s 626ms/step - loss: 0.0187 - accuracy: 0.9933 - val_loss: 1.0899 - val_accuracy: 0.7126
Epoch 7/20
409/409 [==============================] - 260s 636ms/step - loss: 0.0234 - accuracy: 0.9922 - val_loss: 0.6058 -

In [6]:
# Evaluate the model on the test set (you should have a separate test set)
test_generator = datagen.flow_from_directory(
    data_directory,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical',
    classes=['unidentified', 'Hyalomma', 'Rhipicephalus'],
    subset='validation'
)

test_loss, test_accuracy = model.evaluate(test_generator, steps=test_generator.samples // batch_size)
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')

Found 3275 images belonging to 3 classes.
102/102 [==============================] - 71s 700ms/step - loss: 0.7088 - accuracy: 0.8349
Test Accuracy: 83.49%


In [7]:
model.save('tickBugsModelV2.h5')

/opt/homebrew/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [10]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np

# Load the model
loaded_model = load_model('tickBugsModelV2.h5')

# Function to preprocess an input image
def preprocess_image(image_path):
    img = image.load_img(image_path, target_size=(128, 128))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0  # Normalize the image
    return img_array

# Function to make predictions on a given image
def predict_image(model, image_path):
    preprocessed_image = preprocess_image(image_path)
    prediction = model.predict(preprocessed_image)
    class_index = np.argmax(prediction)
    class_labels = ['unidentified', 'Hyalomma', 'Rhipicephalus']
    predicted_class = class_labels[class_index]
    return predicted_class

# Path to the image you want to predict
image_path_to_predict = 'test1.jpg'

# Make a prediction
prediction_result = predict_image(loaded_model, image_path_to_predict)
print(f'The predicted class for the image is: {prediction_result}')

1/1 [==============================] - 0s 229ms/step
The predicted class for the image is: Rhipicephalus
